# Async Python

## A briefing on asynchronous python coding, essential in Agent engineering

Here is a masterful tutorial by you-know-who with exercises and comparisons.

https://chatgpt.com/share/680648b1-b0a0-8012-8449-4f90b540886c

This includes how to run async code from a python module.

### And now some examples:

In [41]:
# Let's define an async function

import asyncio

async def do_some_work(a: int):
    print(f"Starting work-{a}")
    await asyncio.sleep(a+1)
    print(f"Work complete-{a}")


In [15]:
# What will this do?

do_some_work(1)

<coroutine object do_some_work at 0x10b4a7030>

In [16]:
# OK let's try that again!

await do_some_work(1)

Starting work-1
Work complete


In [17]:
# What's wrong with this?

async def do_a_lot_of_work():
    do_some_work(1)
    do_some_work(2)
    do_some_work(3)

await do_a_lot_of_work()

/var/folders/47/xf_9l4nj7vlb1ph69dszbpsw0000gn/T/ipykernel_13615/3501925770.py:4: RuntimeWarning: coroutine 'do_some_work' was never awaited
  do_some_work(1)
/var/folders/47/xf_9l4nj7vlb1ph69dszbpsw0000gn/T/ipykernel_13615/3501925770.py:5: RuntimeWarning: coroutine 'do_some_work' was never awaited
  do_some_work(2)
/var/folders/47/xf_9l4nj7vlb1ph69dszbpsw0000gn/T/ipykernel_13615/3501925770.py:6: RuntimeWarning: coroutine 'do_some_work' was never awaited
  do_some_work(3)


In [42]:
# Interesting warning! Let's fix it

async def do_a_lot_of_work():
    await do_some_work(1)
    await do_some_work(2)
    await do_some_work(3)

await do_a_lot_of_work()

Starting work-1
Work complete-1
Starting work-2
Work complete-2
Starting work-3
Work complete-3


In [35]:
# And now let's do it in parallel
# It's important to recognize that this is not "multi-threading" in the way that you may be used to
# The asyncio library is running on a single thread, but it's using a loop to switch between tasks while one is waiting

async def do_a_lot_of_work_in_parallel():
    await asyncio.gather(do_some_work(1), do_some_work(2), do_some_work(3))

await do_a_lot_of_work_in_parallel()

Starting work-1
Starting work-2
Starting work-3
Work complete-1
Work complete-2
Work complete-3


### Finally - try writing a python module that calls do_a_lot_of_work_in_parallel

See the link at the top; you'll need something like this in your module:

```python
if __name__ == "__main__":
    asyncio.run(do_a_lot_of_work_in_parallel())
```

In [3]:
import asyncio
import time
from concurrent.futures import ThreadPoolExecutor

# --- Simulate a slow, blocking database query ---
def blocking_db_query(request_id: int):
    print(f"[BLOCKING] DB query started for request {request_id}")
    time.sleep(2)  # Simulate blocking operation
    print(f"[BLOCKING] DB query completed for request {request_id}")
    return f"Data for request {request_id}"

# --- Simulate a non-blocking API call ---
async def async_api_call(request_id: int):
    print(f"[ASYNC] API call started for request {request_id}")
    await asyncio.sleep(1)  # Simulated I/O
    print(f"[ASYNC] API call completed for request {request_id}")
    return f"Response for request {request_id}"

# --- Process a single request ---
async def handle_request(request_id: int, executor):
    print(f"[REQUEST {request_id}] Handling started")

    # Step 1: Start an async operation (non-blocking)
    api_task = asyncio.create_task(async_api_call(request_id))

    # Step 2: Run blocking operation in a thread (non-blocking from event loop)
    loop = asyncio.get_running_loop()
    db_future = loop.run_in_executor(executor, blocking_db_query, request_id)

    # Step 3: Wait for both to finish (concurrently)
    api_result, db_result = await asyncio.gather(api_task, db_future)

    print(f"[REQUEST {request_id}] Done with: {api_result} + {db_result}\n")

# --- Simulate a webserver receiving many requests ---
async def main():
    print("Web server started")

    # Use thread pool for blocking functions
    executor = ThreadPoolExecutor(max_workers=3)

    # Simulate 5 incoming requests at once
    tasks = [handle_request(i, executor) for i in range(1, 6)]

    await asyncio.gather(*tasks)
    executor.shutdown()
    print("Web server finished all requests")

if __name__ == "__main__":
    import sys
    if "ipykernel" in sys.modules:
        await main()  # Notebook-friendly
    else:
        asyncio.run(main())  # Script-friendly



Web server started
[REQUEST 1] Handling started
[BLOCKING] DB query started for request 1
[REQUEST 2] Handling started
[BLOCKING] DB query started for request 2
[REQUEST 3] Handling started
[BLOCKING] DB query started for request 3
[REQUEST 4] Handling started
[REQUEST 5] Handling started
[ASYNC] API call started for request 1
[ASYNC] API call started for request 2
[ASYNC] API call started for request 3
[ASYNC] API call started for request 4
[ASYNC] API call started for request 5
[ASYNC] API call completed for request 1
[ASYNC] API call completed for request 2
[ASYNC] API call completed for request 3
[ASYNC] API call completed for request 4
[ASYNC] API call completed for request 5
[BLOCKING] DB query completed for request 2
[BLOCKING] DB query started for request 4
[BLOCKING] DB query completed for request 1
[REQUEST 2] Done with: Response for request 2 + Data for request 2

[BLOCKING] DB query started for request 5
[REQUEST 1] Done with: Response for request 1 + Data for request 1

[B